Analisis Exploratorio de datos

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Importando metricas
from sklearn.metrics import (
    mean_absolute_error, 
    mean_squared_error, 
    r2_score, 
    mean_absolute_percentage_error
)

# Importando algoritmos
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor

import warnings
warnings.filterwarnings('ignore')

Carga y Limpieza de Datos


In [2]:
df = pd.read_csv('data/AH/AmesHousing.csv')

# columna objetivo es 'SalePrice'
target_col = 'SalePrice'

# Separar X y y
X = df.drop(columns=[target_col])
y = df[target_col]

# Convertir variables categoricas a numéricas
X = pd.get_dummies(X, drop_first=True)

# Division de los datos (80% entrenamiento 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Manejo de valores nulos 
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)
# Escalado de datos (lo ocupo para KNN y MLPRegressor)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print(f"Dimensiones del set de entrenamiento: {X_train_scaled.shape}")
print(f"Dimensiones del set de prueba: {X_test_scaled.shape}")

Dimensiones del set de entrenamiento: (2344, 262)
Dimensiones del set de prueba: (586, 262)


Funcion para calcular R2 Ajustado

In [3]:
def calcular_r2_ajustado(r2, n, p):
    """
    Calcula el R-cuadrado ajustado.
    :param r2: Valor de R-cuadrado obtenido del modelo.
    :param n: Número de observaciones (filas) en el set de prueba.
    :param p: Número de variables predictoras (columnas) en el modelo.
    """
    r2_adj = 1 - (1 - r2) * ((n - 1) / (n - p - 1))
    return r2_adj

Entrenamiento y Comparación de Modelos

In [ ]:
# Definicion de los modelos a comparar
modelos = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(random_state=42),
    "K-Neighbors Regressor": KNeighborsRegressor(n_neighbors=5),
    "MLP Regressor": MLPRegressor(random_state=42, max_iter=500)
}

# Obtenemos n y p para el R2 ajustado
n_observaciones = X_test_scaled.shape[0]
p_variables = X_test_scaled.shape[1]

# Diccionario para almacenar los resultados y hacer gráficos posteriormente si se requiere
resultados = {}

print("-" * 50)
print("COMPARACIÓN DE MODELOS DE REGRESIÓN")
print("-" * 50)

for nombre, modelo in modelos.items():
    # Entrenamiento del modelo
    modelo.fit(X_train_scaled, y_train)
    
    # Predicciones
    y_pred = modelo.predict(X_test_scaled)
    
    # Calculo de metricas
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse) # RMSE es la raíz cuadrada del MSE
    r2 = r2_score(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    r2_adj = calcular_r2_ajustado(r2, n_observaciones, p_variables)
    
    # Guardar resultados 
    resultados[nombre] = {
        "MAE": mae, "MSE": mse, "RMSE": rmse, 
        "R2": r2, "MAPE": mape, "R2 Ajustado": r2_adj
    }
    
    # resultados
    print(f"Modelo: {nombre}")
    print(f"  > MAE        : {mae:,.4f}")
    print(f"  > MSE        : {mse:,.4f}")
    print(f"  > RMSE       : {rmse:,.4f}")
    print(f"  > MAPE       : {mape:,.4f}")
    print(f"  > R2         : {r2:,.4f}")
    print(f"  > R2 Ajustado: {r2_adj:,.4f}")
    print("-" * 50)

--------------------------------------------------
COMPARACIÓN DE MODELOS DE REGRESIÓN
--------------------------------------------------
Modelo: Linear Regression
  > MAE        : 15,969.2810
  > MSE        : 834,180,950.6425
  > RMSE       : 28,882.1909
  > MAPE       : 0.0907
  > R2         : 0.8960
  > R2 Ajustado: 0.8116
--------------------------------------------------
Modelo: Random Forest Regressor
  > MAE        : 15,885.3554
  > MSE        : 688,213,593.2201
  > RMSE       : 26,233.8254
  > MAPE       : 0.0856
  > R2         : 0.9142
  > R2 Ajustado: 0.8445
--------------------------------------------------
Modelo: K-Neighbors Regressor
  > MAE        : 26,241.7638
  > MSE        : 1,769,629,324.1720
  > RMSE       : 42,066.9624
  > MAPE       : 0.1387
  > R2         : 0.7793
  > R2 Ajustado: 0.6002
--------------------------------------------------
Modelo: MLP Regressor
  > MAE        : 93,485.4409
  > MSE        : 11,315,362,030.6626
  > RMSE       : 106,373.6905
  > MAPE 